In [1]:
import timeit, tracemalloc
import numpy as np
import pylops
import warnings

In [2]:
def bench(f, *args, number=1, repeat=10):
    """
    Benchmarks function `f(*args)`:
    - number: calls per repeat
    - repeat: number of repeats
    Returns median runtime per call and peak memory usage.
    """
    tracemalloc.start()

    # Run `repeat` times, each running `number` calls
    timings = timeit.repeat(lambda: f(*args), repeat=repeat, number=number)

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Median runtime per call
    median_time = np.median(timings) / number
    print(f"Median time per call: {median_time:.6e} s")
    print(f"Peak memory usage: {peak / 1024:.1f} KB")
    return median_time, peak

In [ ]:
def update_c(
    F,
    latent_dyn,
    params_update_c,
    cofficients,
):
    """
    The function comes to update the coefficients of the sub-dynamics, {c_i}, by solving the inverse or solving lasso.
    Inputs:
        F               = list of sub-dynamics. Should be a list of k X k arrays.
        latent_dyn      = latent_dynamics (dynamics dimensions X time)
        params_update_c = dictionary with keys:
            update_c_type  = options:
                 - 'inv' (least squares)
                 - 'lasso' (sklearn lasso)
                 - 'fista' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.FISTA.html)
                 - 'omp' (https://pylops.readthedocs.io/en/latest/gallery/plot_ista.html#sphx-glr-gallery-plot-ista-py)
                 - 'ista' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.ISTA.html)
                 - 'IRLS' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.IRLS.html)
                 - 'spgl1' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.SPGL1.html)


                 - . Refers to the way the coefficients should be claculated (inv -> no l1 regularization)
            reg_term       = scalar between 0 to 1, describe the reg. term on the cofficients
            smooth_term    = scalar between 0 to 1, describe the smooth term on the cofficients (c_t - c_(t-1))
        direction      = can be c2n (clean to noise) OR  n2c (noise to clean)
        other_params   = additional parameters for the lasso solver (optional)
        random_state   = random state for reproducability (optional)
        skip_error     = whether to skip an error when solving the inverse for c (optional)
        cofficients    = needed only if smooth_term > 0. This is the reference coefficients matrix to apply the constraint (c_hat_t - c_(t-1)) on.

    Outputs:
        coefficients matrix (k X T), type = np.array

    example:
    coeffs = update_c(np.random.rand(2,2), np.random.rand(2,15),{})
    """
    n_times = latent_dyn.shape[1] - 1

    clear_dyn = latent_dyn
    coeffs_list = []

    for time_point in np.arange(n_times):
        cur_dyn = clear_dyn[:, time_point]
        next_dyn = latent_dyn[:, time_point + 1]
        total_next_dyn = next_dyn
        f_x_mat = []
        for f_i in F:
            f_x_mat.append(f_i @ cur_dyn)
        stacked_fx = np.vstack(f_x_mat).T
        stacked_fx[stacked_fx > 10**8] = 10**8

        if params_update_c["smooth_term"] > 0 and time_point > 0:
            if len(cofficients) == 0:
                warnings.warn(
                    "Warning: you called the smoothing option without defining coefficients"
                )
        if (
            params_update_c["smooth_term"] > 0
            and time_point > 0
            and len(cofficients) > 0
        ):
            c_former = cofficients[:, time_point - 1]
            total_next_dyn_full = np.hstack(
                [total_next_dyn, np.sqrt(params_update_c["smooth_term"]) * c_former]
            )
            stacked_fx_full = np.vstack(
                [
                    stacked_fx,
                    np.sqrt(params_update_c["smooth_term"]) * np.eye(F.shape[0]),
                ]
            )
        else:
            total_next_dyn_full = total_next_dyn
            stacked_fx_full = stacked_fx

        Aop = pylops.MatrixMult(stacked_fx_full)
        if "threshkind" not in params_update_c:
            params_update_c["threshkind"] = "soft"

        coeffs = pylops.optimization.sparsity.fista(
            Aop,
            total_next_dyn_full.flatten(),
            niter=10,
            eps=params_update_c["reg_term"],
            threshkind=params_update_c.get("threshkind"),
        )[0]

        coeffs_list.append(coeffs.flatten())
    coeffs_final = np.vstack(coeffs_list)

    return coeffs_final.T

In [54]:
num_latents = 30
num_timepoints = 10000
num_motifs = 30

X = np.random.randn(num_latents, num_timepoints)
c = np.random.randn(num_motifs, num_timepoints - 1)
F = np.random.randn(num_motifs, num_latents, num_latents)

# coeffs = update_c(F, X, {"smooth_term": 0.01, "reg_term": 0.01})

bench(update_c, F, X, {"smooth_term": 0.01, "reg_term": 0.01}, c, number=1, repeat=5)


Median time per call: 4.873437e+01 s
Peak memory usage: 184440.7 KB


(np.float64(48.7343733930029), 188867320)

In [7]:
c.shape

(3, 999)